# EDA

### Dicionário de Dados de Voos (`flights.csv`)

Abaixo estão as descrições das variáveis contidas no conjunto de dados de voos, incluindo seus tipos e unidades de medida.

| Coluna | Descrição | Tipo / Unidade |
| :--- | :--- | :--- |
| **YEAR** | Ano do voo (ex.: 2015) | Inteiro |
| **MONTH** | Mês do voo (1 a 12) | Inteiro |
| **DAY** | Dia do mês do voo (1 a 31) | Inteiro |
| **DAY_OF_WEEK** | Dia da semana (1 = Segunda, 7 = Domingo) | Inteiro |
| **AIRLINE** | Código da companhia aérea (ex.: AA = American Airlines) | Categórica |
| **FLIGHT_NUMBER** | Número do voo | Inteiro |
| **TAIL_NUMBER** | Número de registro da aeronave | Texto |
| **ORIGIN_AIRPORT** | Código IATA do aeroporto de origem (ex.: ATL) | Categórica |
| **DESTINATION_AIRPORT** | Código IATA do aeroporto de destino | Categórica |
| **SCHEDULED_DEPARTURE** | Horário de partida programado | HHMM (Inteiro) |
| **DEPARTURE_TIME** | Horário real de partida | HHMM (Inteiro) |
| **DEPARTURE_DELAY** | Atraso na partida | Numérico (Minutos) |
| **TAXI_OUT** | Tempo gasto taxiando até a decolagem | Numérico (Minutos) |
| **WHEELS_OFF** | Horário em que o avião decolou | HHMM (Inteiro) |
| **SCHEDULED_TIME** | Tempo total programado de voo | Numérico (Minutos) |
| **ELAPSED_TIME** | Tempo total real de voo | Numérico (Minutos) |
| **AIR_TIME** | Tempo no ar | Numérico (Minutos) |
| **DISTANCE** | Distância entre origem e destino | Numérico (Milhas) |
| **WHEELS_ON** | Horário em que as rodas tocaram o solo | HHMM (Inteiro) |
| **TAXI_IN** | Tempo taxiando até o portão de desembarque | Numérico (Minutos) |
| **SCHEDULED_ARRIVAL** | Horário de chegada programado | HHMM (Inteiro) |
| **ARRIVAL_TIME** | Horário de chegada real | HHMM (Inteiro) |
| **ARRIVAL_DELAY** | Atraso na chegada | Numérico (Minutos) |
| **DIVERTED** | Indica se o voo foi desviado (1 = Sim, 0 = Não) | Binária |
| **CANCELLED** | Indica se o voo foi cancelado (1 = Sim, 0 = Não) | Binária |
| **CANCELLATION_REASON** | Motivo do cancelamento (A=Airline, B=Weather, C=NAS, D=Security) | Categórica |
| **AIR_SYSTEM_DELAY** | Atraso causado por controle de tráfego aéreo | Numérico (Minutos) |
| **SECURITY_DELAY** | Atraso causado por problemas de segurança | Numérico (Minutos) |
| **AIRLINE_DELAY** | Atraso causado pela companhia aérea | Numérico (Minutos) |
| **LATE_AIRCRAFT_DELAY** | Atraso causado por chegada tardia da aeronave | Numérico (Minutos) |
| **WEATHER_DELAY** | Atraso causado por condições meteorológicas | Numérico (Minutos) |


### Dicionário de Dados de Clima (NOAA GHCN-Daily)

| Variável | Nome | Descrição | Unidade (Pós-Processamento) |
| :--- | :--- | :--- | :--- |
| **PRCP** | Precipitação | Quantidade total de chuva ou neve derretida em 24h. | Milímetros (mm) |
| **SNOW** | Neve total | Quantidade total de neve que caiu em 24h. | Milímetros (mm) |
| **SNWD** | Profundidade da Neve | Altura da neve acumulada no solo em determinado momento. | Milímetros (mm) |
| **TMAX** | Temperatura Máxima | A maior temperatura registrada durante o dia. | Graus Celsius (°C) |
| **TMIN** | Temperatura Mínima | A menor temperatura registrada durante o dia. | Graus Celsius (°C) |
| **AWND** | Vento Médio | Velocidade média diária do vento. | Metros por segundo (m/s) |
| **WSF2** | Rajada Máxima | Velocidade da rajada de vento mais rápida (média de 2 min). | Metros por segundo (m/s) |
| **WT01** | Nevoeiro | Indicador se houve neblina ou nevoeiro no dia. | Binário (1 = Sim, 0 = Não) |

In [1]:
#importando bibliotecas
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff 
from plotly.subplots import make_subplots

import pandas as pd
from scipy.spatial import cKDTree

pd.set_option('display.max_columns', None)

In [2]:
#importando dados
df_flights = pd.read_csv('../data/raw/flights.csv')
df_airlines = pd.read_csv('../data/raw/airlines.csv')
df_airports = pd.read_csv('../data/raw/airports.csv')
df_weather = pd.read_csv(r'../data/raw/us_weather.csv')
df_stations = pd.read_csv(r'../data/raw/stations.csv')

/tmp/ipykernel_230065/2170366039.py:2: DtypeWarning: Columns (0: ORIGIN_AIRPORT, 1: DESTINATION_AIRPORT) have mixed types. Specify dtype option on import or set low_memory=False.
  df_flights = pd.read_csv('../data/raw/flights.csv')


### Tratamento de dados

In [3]:
df_airports.columns

Index(['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY', 'LATITUDE',
       'LONGITUDE'],
      dtype='str')

In [4]:
df = df_flights.merge(
    df_airlines.rename(columns={'AIRLINE': 'AIRLINE_NAME'}), 
    left_on='AIRLINE', right_on='IATA_CODE', how='left'
).drop(columns=['IATA_CODE'])

df_airports = df_airports.rename(columns={'AIRPORT': 'AIRPORT_NAME'})

df = df.merge(
    df_airports.add_prefix('ORIGIN_'), 
    left_on='ORIGIN_AIRPORT', right_on='ORIGIN_IATA_CODE', how='left'
).drop(columns=['ORIGIN_IATA_CODE'])

df = df.merge(
    df_airports.add_prefix('DEST_'),
    left_on='DESTINATION_AIRPORT', right_on='DEST_IATA_CODE', how='left'
).drop(columns=['DEST_IATA_CODE'])

df.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,AIRLINE_NAME,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,ORIGIN_COUNTRY,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,DEST_AIRPORT_NAME,DEST_CITY,DEST_STATE,DEST_COUNTRY,DEST_LATITUDE,DEST_LONGITUDE
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,2354.0,-11.0,21.0,15.0,205.0,194.0,169.0,1448,404.0,4.0,430,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Alaska Airlines Inc.,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,2.0,-8.0,12.0,14.0,280.0,279.0,263.0,2330,737.0,4.0,750,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Palm Beach International Airport,West Palm Beach,FL,USA,26.68316,-80.09559
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,18.0,-2.0,16.0,34.0,286.0,293.0,266.0,2296,800.0,11.0,806,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,US Airways Inc.,San Francisco International Airport,San Francisco,CA,USA,37.61900,-122.37484,Charlotte Douglas International Airport,Charlotte,NC,USA,35.21401,-80.94313
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,15.0,-5.0,15.0,30.0,285.0,281.0,258.0,2342,748.0,8.0,805,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Miami International Airport,Miami,FL,USA,25.79325,-80.29056
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,24.0,-1.0,11.0,35.0,235.0,215.0,199.0,1448,254.0,5.0,320,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Alaska Airlines Inc.,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619


In [5]:
#criar uma lista de TODOS os aeroportos
df_airports_temp = df[['ORIGIN_AIRPORT', 'ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE']].drop_duplicates().dropna()

#criando a árvore de busca
tree = cKDTree(df_stations[['LATITUDE', 'LONGITUDE']])

#encontrando a estação mais próxima para cada aeroporto único
dist, indices = tree.query(df_airports_temp[['ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE']])

#tabela de mapeamento: iata -> station_id
df_mapping = pd.DataFrame({
    'IATA_CODE': df_airports_temp['ORIGIN_AIRPORT'].values,
    'STATION_ID': df_stations.iloc[indices]['STATION'].values
})

#adicionando id da estação de origem
df = df.merge(df_mapping, left_on='ORIGIN_AIRPORT', right_on='IATA_CODE', how='left')
df.rename(columns={'STATION_ID': 'STATION_ORIGIN'}, inplace=True)
df.drop(columns=['IATA_CODE'], inplace=True)

#tratando dadas para o merge
df['DATE'] = pd.to_datetime(df[['YEAR', 'MONTH', 'DAY']])
df_weather['DATE'] = pd.to_datetime(df_weather['DATE'])

df = df.merge(
    df_weather.add_prefix('ORIGIN_'), 
    left_on=['STATION_ORIGIN', 'DATE'], 
    right_on=['ORIGIN_STATION', 'ORIGIN_DATE'], 
    how='left'
)

#remover colunas redundantes criadas pelos prefixos
cols_to_drop = ['ORIGIN_STATION', 'ORIGIN_DATE']
df.drop(columns=cols_to_drop, inplace=True)

print(df.shape[0])
df.head()

5819079


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,AIRLINE_NAME,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,ORIGIN_COUNTRY,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,DEST_AIRPORT_NAME,DEST_CITY,DEST_STATE,DEST_COUNTRY,DEST_LATITUDE,DEST_LONGITUDE,STATION_ORIGIN,DATE,ORIGIN_AWND,ORIGIN_PRCP,ORIGIN_SNOW,ORIGIN_SNWD,ORIGIN_TMAX,ORIGIN_TMIN,ORIGIN_WSF2,ORIGIN_WT01
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,2354.0,-11.0,21.0,15.0,205.0,194.0,169.0,1448,404.0,4.0,430,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Alaska Airlines Inc.,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931,USC00500275,2015-01-01,0.0,0.0,0.0,10.2,0.6,-3.3,0.0,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,2.0,-8.0,12.0,14.0,280.0,279.0,263.0,2330,737.0,4.0,750,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Palm Beach International Airport,West Palm Beach,FL,USA,26.68316,-80.09559,USW00023174,2015-01-01,2.3,0.0,NaN,NaN,13.9,2.2,5.4,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,18.0,-2.0,16.0,34.0,286.0,293.0,266.0,2296,800.0,11.0,806,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,US Airways Inc.,San Francisco International Airport,San Francisco,CA,USA,37.61900,-122.37484,Charlotte Douglas International Airport,Charlotte,NC,USA,35.21401,-80.94313,USW00023234,2015-01-01,3.8,0.0,NaN,NaN,12.8,4.4,8.9,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,15.0,-5.0,15.0,30.0,285.0,281.0,258.0,2342,748.0,8.0,805,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,American Airlines Inc.,Los Angeles International Airport,Los Angeles,CA,USA,33.94254,-118.40807,Miami International Airport,Miami,FL,USA,25.79325,-80.29056,USW00023174,2015-01-01,2.3,0.0,NaN,NaN,13.9,2.2,5.4,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,24.0,-1.0,11.0,35.0,235.0,215.0,199.0,1448,254.0,5.0,320,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Alaska Airlines Inc.,Seattle-Tacoma International Airport,Seattle,WA,USA,47.44898,-122.30931,Ted Stevens Anchorage International Airport,Anchorage,AK,USA,61.17432,-149.99619,USW00024233,2015-01-01,1.2,0.0,0.0,0.0,5.6,-3.2,4.0,NaN


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 54 columns):
 #   Column               Dtype         
---  ------               -----         
 0   YEAR                 int64         
 1   MONTH                int64         
 2   DAY                  int64         
 3   DAY_OF_WEEK          int64         
 4   AIRLINE              str           
 5   FLIGHT_NUMBER        int64         
 6   TAIL_NUMBER          str           
 7   ORIGIN_AIRPORT       object        
 8   DESTINATION_AIRPORT  object        
 9   SCHEDULED_DEPARTURE  int64         
 10  DEPARTURE_TIME       float64       
 11  DEPARTURE_DELAY      float64       
 12  TAXI_OUT             float64       
 13  WHEELS_OFF           float64       
 14  SCHEDULED_TIME       float64       
 15  ELAPSED_TIME         float64       
 16  AIR_TIME             float64       
 17  DISTANCE             int64         
 18  WHEELS_ON            float64       
 19  TAXI_IN              float64    

In [7]:
cols = [
    'ORIGIN_PRCP', 'ORIGIN_SNOW', 'ORIGIN_SNWD', 'ORIGIN_WT01',
    'ORIGIN_AWND', 'ORIGIN_TMAX', 'ORIGIN_TMIN', 'ORIGIN_WSF2'
]
df_unmatched = df.dropna(subset=cols)
round((df_unmatched.shape[0] / df.shape[0]) * 100, 2)

21.46

Tratamento de nulos

In [ ]:
nulos = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    '%': (df.isnull().sum() / len(df) * 100).round(2)
})

print(nulos[nulos['Nulos'] > 0])

                       Nulos      %
TAIL_NUMBER            14721   0.25
DEPARTURE_TIME         86153   1.48
DEPARTURE_DELAY        86153   1.48
TAXI_OUT               89047   1.53
WHEELS_OFF             89047   1.53
SCHEDULED_TIME             6   0.00
ELAPSED_TIME          105071   1.81
AIR_TIME              105071   1.81
WHEELS_ON              92513   1.59
TAXI_IN                92513   1.59
ARRIVAL_TIME           92513   1.59
ARRIVAL_DELAY         105071   1.81
CANCELLATION_REASON  5729195  98.46
AIR_SYSTEM_DELAY     4755640  81.72
SECURITY_DELAY       4755640  81.72
AIRLINE_DELAY        4755640  81.72
LATE_AIRCRAFT_DELAY  4755640  81.72
WEATHER_DELAY        4755640  81.72
ORIGIN_AIRPORT_NAME   486165   8.35
ORIGIN_CITY           486165   8.35
ORIGIN_STATE          486165   8.35
ORIGIN_COUNTRY        486165   8.35
ORIGIN_LATITUDE       490770   8.43
ORIGIN_LONGITUDE      490770   8.43
DEST_AIRPORT_NAME     486165   8.35
DEST_CITY             486165   8.35
DEST_STATE            486165

: 

In [ ]:
df_curated = df.copy()

In [ ]:
#excluindo voos sem dados climáticos
df_curated = df_curated[~df_curated.index.isin(df_unmatched.index)]

In [ ]:
df_curated.shape[0]

4570438

In [ ]:
#preenchendo campos nulos que ocorrem por não haver atraso
delay_cols = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']
df_curated[delay_cols] = df_curated[delay_cols].fillna(0)

#removendo motivo de cancelamento, praticamente vazia
df_curated = df_curated.drop(columns=['CANCELLATION_REASON'])

#removendo demais nulos
df_curated = df_curated.dropna(subset=[
    'ARRIVAL_DELAY', 
    'ORIGIN_CITY', 
    'DEST_CITY', 
    'ORIGIN_LATITUDE', 
    'DEST_LATITUDE',
    'TAIL_NUMBER'
])

In [ ]:
#colunas assumidas como 0 dado a não ocorrência do evento
cols_to_zero = [
    'ORIGIN_PRCP', 'ORIGIN_SNOW', 'ORIGIN_SNWD', 'ORIGIN_WT01'
]
df_curated[cols_to_zero] = df_curated[cols_to_zero].fillna(0)

#colunas assumidas como média dado a ocorrência do evento 
cols_to_mean = [
    'ORIGIN_AWND', 'ORIGIN_TMAX', 'ORIGIN_TMIN', 'ORIGIN_WSF2'
]
for col in cols_to_mean:
    df_curated[col] = df_curated[col].fillna(df_curated.groupby(['ORIGIN_AIRPORT', 'MONTH'])[col].transform('mean'))
    df_curated[col] = df_curated[col].fillna(df_curated[col].mean())

In [ ]:
nulos = pd.DataFrame({
    'Nulos': df_curated.isnull().sum(),
    '%': (df_curated.isnull().sum() / len(df_curated) * 100).round(2)
})

print(nulos[nulos['Nulos'] > 0])

Empty DataFrame
Columns: [Nulos, %]
Index: []


Filtrando dados

In [ ]:
#removendo voos cancelados e desviados, não ajudam na classificação
df_curated = df_curated[(df_curated['CANCELLED'] == 0) & (df_curated['DIVERTED'] == 0)]


Obtendo target

In [ ]:
#criando a variável target
df_curated['IS_DELAYED'] = (df_curated['ARRIVAL_DELAY'] > 15).astype(int)

In [ ]:
df_curated.head()

Exportando dados tratados

In [ ]:
df_curated.to_csv('../data/curated/data.csv', index=False)
df_curated.to_pickle('../data/curated/data.pkl')

: 